# 02 — The decision boundary & reading the weights

*Notebook 2 of 6 — Logistic Regression*

In NB 1 a single feature became a probability. Now we hand the model **two** features, give each its
own weight, and learn to read the geometry that results: where the model splits the classes, and what
the weights are telling us.

**Prerequisites.** NB 1: the sigmoid, odds and log-odds, and the fact that the score *is* the
log-odds. Module 00: the feature space (NB 02), the nearest-centroid classifier (NB 05), and
standardization fitted on training data (NB 11). Chapter 01: the scale trap (NB 02).

**What you'll be able to do.**
- Write the two-feature score $z = w_1 x_1 + w_2 x_2 + b$ and find its **decision boundary**.
- Explain why the weight vector $w$ is **perpendicular** to that boundary and why its length sets the
  steepness.
- Read each weight as a **change in log-odds per standardized unit** — its sign a direction, its size
  a strength.
- Contrast this **weighted** boundary with the **unweighted** nearest-centroid bisector.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)  # no randomness here, but we keep the habit

# The course fil-rouge: Adelie vs Gentoo. Two features now: bill length and flipper length (mm).
df = datasets.load_penguins()
X_df, y = datasets.penguins_xy(df)
print(y.value_counts())
df.head()

## Where we are

NB 1 took **one** feature, `bill_length_mm`, turned it into a score, and squashed that score into
$P(\text{Gentoo})$ with the sigmoid. One feature gave a single dividing point on a line.

We have drawn a boundary before. In module 00, the **nearest-centroid** classifier (NB 05) put a
penguin in whichever class had the closer average — its boundary was the **perpendicular bisector**
between the two class means, and it treated both features the same way.

Logistic regression keeps the boundary but makes it **weighted**: each feature gets its own weight, so
one feature can count more than another. This notebook draws that weighted line and, as importantly,
learns to **read** what the weights mean. We are still setting the weights **by hand** — NB 3 and NB 4
will *find* them.

## First, put the features on the same scale

Bill length runs from about 32 to 60 mm; flipper length from about 170 to 230 mm. The two live on very
different scales. In chapter 01 (NB 02) that mismatch was the **scale trap**: a distance was dominated
by whichever feature had the larger range. Here the danger is subtler but related — we want to compare
weight *magnitudes* ("which feature moves the odds more?"), and that comparison is only fair when the
features share a scale.

So we **standardize** each feature to a z-score: subtract its mean, divide by its standard deviation. A
value of $+1$ then means "one standard deviation above average", in the same currency for both
features. (There is no train/test split yet; NB 6 will standardize on training data only.)

In [ ]:
feature_names = list(X_df.columns)
X = X_df.to_numpy(dtype=float)

scaler = StandardScaler()
Xs = scaler.fit_transform(X)  # z-scores; a numpy array under the hood

print("means (mm):", scaler.mean_.round(2))
print("std   (mm):", scaler.scale_.round(2))
print("after standardizing -> mean:", Xs.mean(axis=0).round(3), " std:", Xs.std(axis=0).round(3))

pd.DataFrame(Xs, columns=[f"{n} (std)" for n in feature_names]).head()

## A score from two features

With two features the score is still a single number — a weighted sum, one weight per feature, plus the
offset:

$$ z = w_1\, x_1 + w_2\, x_2 + b. $$

Here $x_1$ is the standardized bill length and $x_2$ the standardized flipper length. Everything from
NB 1 still holds: $z$ is the **log-odds**, the sigmoid turns it into $P(\text{Gentoo}) = \sigma(z)$, and
we predict **Gentoo when $P \ge \tfrac12$**, which is exactly **when $z \ge 0$**.

The one new thing is that we now have **two** weights to choose, and they will turn out to carry a
clear meaning.

In [ ]:
def sigmoid(z):
    """Map any real score z to a probability in (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


# Hand-chosen weights (NOT fitted): flipper weighted twice as much as bill, to see the geometry.
# NB 3-4 will *find* the best weights; here we set them to read what weights mean.
w = np.array([1.0, 2.0])  # [bill, flipper], in standardized units
b = 0.0

is_gentoo = (y.to_numpy() == "Gentoo").astype(int)
z = Xs @ w + b
P = sigmoid(z)

# Three penguins: a clear Adelie, a clear Gentoo, and a borderline one near the boundary.
clear_adelie = int(np.argmin(P))
clear_gentoo = int(np.argmax(P))
borderline = int(np.argmin(np.abs(P - 0.5)))
idx = [clear_adelie, borderline, clear_gentoo]
pd.DataFrame(
    {
        "bill_length_mm": X[idx, 0],
        "flipper_length_mm": X[idx, 1],
        "score z": z[idx],
        "P(Gentoo)": P[idx],
        "true species": y.to_numpy()[idx],
    }
).round(3)

## The decision boundary is the line $z = 0$

Three penguins, three honest answers: the short-billed bird reads $P \approx 0$ (Adélie), the
long-billed one $P \approx 1$ (Gentoo), and the borderline bird near the middle reads close to one half
— the model saying "I cannot tell". So **where**, exactly, does it tip from one class to the other?
Where $P = \tfrac12$, which means $z = 0$:

$$ w_1 x_1 + w_2 x_2 + b = 0. $$

In two features that equation is a **straight line** — the **decision boundary**. On one side $z > 0$
and we predict Gentoo; on the other $z < 0$ and we predict Adélie. Two facts about this line are worth
*seeing*, and we will:

- the weight vector $w = (w_1, w_2)$ is **perpendicular** to the boundary — it points straight across
  it, toward the Gentoo side;
- the **length** $\lVert w \rVert$ sets how **steeply** the probability swings from 0 to 1 as you cross
  (the 1-D slope of NB 1, now pointing in a direction).

In [ ]:
# Accuracy of the hand-chosen rule (nothing fitted).
w_norm = np.linalg.norm(w)
pred = (P >= 0.5).astype(int)
accuracy = (pred == is_gentoo).mean()
print(f"Hand-chosen rule accuracy on the {len(P)} penguins: {accuracy:.4f}")
print(f"||w|| = {w_norm:.2f}")

# A grid over the standardized feature plane, coloured by P(Gentoo) = sigmoid(z).
g1 = np.linspace(-2.6, 3.6, 300)
g2 = np.linspace(-2.6, 2.6, 300)
xx, yy = np.meshgrid(g1, g2)
PP = sigmoid(w[0] * xx + w[1] * yy + b)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [2, 1]})

# Left: the cloud, the probability shading, the boundary line, and the w arrow.
shade = axL.contourf(xx, yy, PP, levels=20, cmap=colors.CMAP_PROBA, alpha=0.75)
fig.colorbar(shade, ax=axL, label="P(Gentoo)")
for cls, name in [(0, "Adelie"), (1, "Gentoo")]:
    m = is_gentoo == cls
    axL.scatter(
        Xs[m, 0],
        Xs[m, 1],
        color=colors.CLASS_CYCLE[cls],
        alpha=0.75,
        edgecolor=colors.COLORS["text"],
        linewidth=0.4,
        s=32,
        label=name,
    )
# Boundary line z = 0  ->  x2 = -(w1 x1 + b) / w2
edge = np.array([g1.min(), g1.max()])
axL.plot(
    edge,
    -(w[0] * edge + b) / w[1],
    color=colors.COLORS["text"],
    linewidth=2,
    label="boundary z = 0",
)
# w arrow: the origin lies on the boundary (b = 0), so anchor it there, pointing toward Gentoo.
tip = (w / w_norm) * 1.4
axL.annotate(
    "",
    xy=tip,
    xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=colors.COLORS["highlight"], linewidth=2.5),
)
axL.text(
    tip[0] + 0.07, tip[1], "w", color=colors.COLORS["highlight"], fontsize=13, fontweight="bold"
)
axL.set_xlabel("bill_length (standardized)")
axL.set_ylabel("flipper_length (standardized)")
axL.set_title("The weighted decision boundary")
axL.legend(loc="lower right")
axL.set_xlim(g1.min(), g1.max())
axL.set_ylim(g2.min(), g2.max())
axL.set_aspect("equal", adjustable="box")  # equal axes -> w renders truly perpendicular

# Right: the two weights as bars.
axR.bar(["bill", "flipper"], np.abs(w), color=colors.COLORS["model"])
for i, wi in enumerate(w):
    axR.text(
        i, abs(wi) + 0.05, f"{wi:g}", ha="center", color=colors.COLORS["text"], fontweight="bold"
    )
axR.set_ylabel("|weight| (log-odds per std unit)")
axR.set_title("How much each feature counts")
axR.set_ylim(0, 2.6)

plt.tight_layout()
plt.show()

**Read the figure.** *Left:* the dark line is the **decision boundary** — every penguin on its
upper-right side is predicted Gentoo (the shading climbs toward 1), every penguin on the lower-left side
Adélie (shading toward 0). The band of middle colour hugging the line is where the model is **unsure**,
with $P$ near one half. With these gentle weights that band is wide — more than a third of the penguins
fall in it — yet the rule is still right **98.9%** of the time: being *unsure* (say $P = 0.3$) is not
the same as being *wrong* — though the three penguins it does miss all sit here, inside this band.
The rose **arrow** is the weight vector $w$: it crosses the boundary at a
**right angle** and points toward Gentoo, and its length is the model's confidence slope — short here, so the colour drifts gently from white
(Adélie) to blue (Gentoo) across a wide band rather than snapping at the line. *Right:* the
**flipper** bar is twice the **bill** bar, because we set $w_2 = 2\,w_1$.

## What a weight means: a change in log-odds

Because the score **is** the log-odds (NB 1), each weight has a precise reading. Raise a standardized
feature by **one unit** — one standard deviation, about **5.2 mm** of bill or **15 mm** of flipper — and
the score changes by exactly that feature's weight. So the weight is the **change in log-odds per
standardized unit**:

$$ \Delta z = w_j \quad\Longleftrightarrow\quad \text{odds} \times e^{w_j}. $$

With our weights, one extra standardized unit of **bill** multiplies the odds of Gentoo by
$e^{1} \approx 2.7$; one extra unit of **flipper** multiplies them by $e^{2} \approx 7.4$. The **sign**
is a direction (positive pushes toward Gentoo, negative toward Adélie) and the **size** is a strength.
This reading is fair only because we standardized — otherwise a large weight might reflect nothing more
than a feature measured in small units (the scale trap, again).

## A weighted line versus an unweighted one

Recall the nearest-centroid classifier from module 00. Its boundary is the **perpendicular bisector** of
the two class means: a penguin is Gentoo if it is closer to the Gentoo average than to the Adélie
average. That boundary's direction is fixed by the line joining the two centroids, and it weights the
features by how far apart the centroids are — here, **almost equally**.

Logistic regression is free to weight the features **unequally**. We gave flipper twice the weight of
bill, so our boundary is **tilted** away from the nearest-centroid bisector. Let us put the two lines on
the same plot and measure the tilt.

In [ ]:
# Class centroids in standardized space.
mu_adelie = Xs[is_gentoo == 0].mean(axis=0)
mu_gentoo = Xs[is_gentoo == 1].mean(axis=0)

# Nearest-centroid boundary: the perpendicular bisector of the two centroids.
nc_normal = mu_gentoo - mu_adelie       # points from Adelie toward Gentoo
mid = 0.5 * (mu_gentoo + mu_adelie)     # the bisector passes through the midpoint
nc_pred = (((Xs - mid) @ nc_normal) > 0).astype(int)
nc_accuracy = (nc_pred == is_gentoo).mean()

# Angle between w and the centroid difference = the tilt between the two boundary lines.
cos_angle = (w @ nc_normal) / (np.linalg.norm(w) * np.linalg.norm(nc_normal))
tilt_deg = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
print(f"hand-logistic accuracy:    {accuracy:.4f}")
print(f"nearest-centroid accuracy: {nc_accuracy:.4f}")
print(f"tilt between the two boundaries: {tilt_deg:.1f} degrees")

edge = np.array([-2.6, 3.6])
fig, ax = plt.subplots(figsize=(7, 5.5))
for cls, name in [(0, "Adelie"), (1, "Gentoo")]:
    m = is_gentoo == cls
    ax.scatter(
        Xs[m, 0],
        Xs[m, 1],
        color=colors.CLASS_CYCLE[cls],
        alpha=0.6,
        edgecolor=colors.COLORS["text"],
        linewidth=0.4,
        s=32,
        label=name,
    )
# Logistic (hand) boundary: w1 x1 + w2 x2 = 0
ax.plot(
    edge,
    -(w[0] * edge) / w[1],
    color=colors.COLORS["model"],
    linewidth=2.5,
    label="logistic (weighted)",
)
# Nearest-centroid bisector: nc_normal . (x - mid) = 0
ax.plot(
    edge,
    (nc_normal @ mid - nc_normal[0] * edge) / nc_normal[1],
    color=colors.COLORS["highlight"],
    linewidth=2.5,
    linestyle="--",
    label="nearest-centroid (unweighted)",
)
ax.scatter(
    *mu_adelie,
    color=colors.CLASS_CYCLE[0],
    marker="X",
    s=220,
    edgecolor=colors.COLORS["text"],
    linewidth=1.2,
    zorder=5,
)
ax.scatter(
    *mu_gentoo,
    color=colors.CLASS_CYCLE[1],
    marker="X",
    s=220,
    edgecolor=colors.COLORS["text"],
    linewidth=1.2,
    zorder=5,
)
ax.set_xlabel("bill_length (standardized)")
ax.set_ylabel("flipper_length (standardized)")
ax.set_title("Two boundaries on the same data")
ax.legend(loc="lower right")
ax.set_xlim(-2.6, 3.6)
ax.set_ylim(-2.6, 2.6)
ax.set_aspect("equal", adjustable="box")  # equal axes -> the tilt reads truly
plt.show()

**Read the figure.** Same penguins, two boundaries. The large **X** marks are the class averages; the
dashed rose line is the nearest-centroid bisector, fixed by the line between them. The blue line is our
weighted logistic boundary — **tilted about 16°** because we let flipper count more than bill. On *this*
near-separable data the two score almost the same (nearest-centroid **99.3%**, our hand weights
**98.9%**), so the tilt can look like a detail. But logistic regression has bought two things
nearest-centroid cannot give: a **probability** at every point (the shading of the previous figure), and
**interpretable weights** that say which feature matters and by how much. On data where the features
deserve unequal weight, that freedom to tilt is the difference between a good boundary and a poor
one.

## Turning the line and sliding it

Two knobs shape the boundary, and they do different jobs:

- changing the **ratio** of the weights **rotates** the line (more weight on a feature turns the
  boundary to separate along it);
- changing the bias $b$ **slides** the line parallel to itself, without rotating it (the offset moves
  the half-way crossing).

These are the dials we will stop guessing and start **optimizing** in NB 3 and NB 4.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.8))
edge = np.array([-2.6, 3.6])

# Faint cloud behind both panels.
for ax in (axL, axR):
    for cls in (0, 1):
        m = is_gentoo == cls
        ax.scatter(Xs[m, 0], Xs[m, 1], color=colors.CLASS_CYCLE[cls], alpha=0.18, s=20)
    ax.set_xlim(-2.6, 3.6)
    ax.set_ylim(-2.6, 2.6)
    ax.set_aspect("equal", adjustable="box")  # equal axes -> rotation reads truly
    ax.set_xlabel("bill_length (standardized)")

# Left: rotate by changing w (double the bill weight).
axL.plot(edge, -(1.0 * edge) / 2.0, color=colors.COLORS["model"], linewidth=2.5, label="w = (1, 2)")
axL.plot(
    edge,
    -(2.0 * edge) / 2.0,
    color=colors.COLORS["highlight"],
    linewidth=2.5,
    linestyle="--",
    label="w = (2, 2)  double bill",
)
axL.set_ylabel("flipper_length (standardized)")
axL.set_title("Rotate: change the weights")
axL.legend(loc="lower right")

# Right: shift by changing b (parallel lines).
for bias, style in [(-2.0, ":"), (0.0, "-"), (2.0, "--")]:
    axR.plot(
        edge,
        -(1.0 * edge + bias) / 2.0,
        color=colors.COLORS["model"],
        linewidth=2,
        linestyle=style,
        label=f"b = {bias:+.0f}",
    )
axR.set_title("Shift: change the bias b")
axR.legend(loc="lower right")

plt.tight_layout()
plt.show()

**Read the figure.** *Left:* doubling the bill weight (from 1 to 2) gives bill an equal say and
**rotates** the boundary toward the diagonal — toward the equal-weighting line that nearest-centroid
drew. *Right:* changing $b$ leaves the angle untouched and **slides** the line across the cloud; raising
$b$ moves the half-way crossing so that more penguins fall on the Gentoo side. **Weights turn the line;
the bias moves it.**

## Your turn

1. **Which side?** Take a penguin at standardized coordinates $(x_1, x_2) = (0.5, 1.0)$ — a slightly
   long bill and a long flipper. Compute $z = w_1 x_1 + w_2 x_2 + b$ with $w = (1, 2)$, $b = 0$. Is it
   positive or negative? Which species does the model predict, and how confident is it
   ($P = \sigma(z)$)?
2. **Turn the line.** Set `w = np.array([2.0, 2.0])` (double the bill weight) and rerun the Figure A
   cell. Which way does the boundary rotate, and which feature now moves the odds more?
3. **Place the line.** Keep $w = (1, 2)$. Find the bias $b$ that makes the boundary pass through the
   point $(x_1, x_2) = (1, 1)$. (Set $z = 0$ there and solve for $b$.) Check it by plugging the point
   back in.

## What you built

- The two-feature score $z = w_1 x_1 + w_2 x_2 + b$ and its **decision boundary**, the line $z = 0$
  where $P = \tfrac12$.
- Why the weight vector $w$ is **perpendicular** to the boundary, and why $\lVert w \rVert$ sets its
  **steepness**.
- How to **read a weight**: a change in log-odds per standardized unit — multiply the odds by $e^{w_j}$
  — with the sign a direction and the size a strength.
- The contrast between a **weighted** logistic boundary and the **unweighted** nearest-centroid
  bisector, and how weights **rotate** the line while the bias **shifts** it.

**Vocabulary.** *decision boundary* · *weight vector $w$* · *normal / perpendicular* ·
*$\lVert w \rVert$ (steepness)* · *bias $b$* · *weight as a log-odds contribution* ·
*standardized units*.

You can now draw the line a logistic model uses and read its weights out loud. What we still do by hand
is **choose** those weights — and that is exactly what the next two notebooks fix.

## Going further (optional)

With more than two features the boundary is no longer a line but a flat **hyperplane**
$w \cdot x + b = 0$ — the same idea, one weight per feature, with $w$ still perpendicular to it. The
catch is that the boundary is always **flat**: when the true separation between classes is curved, a
straight boundary **underfits** it. Naming the limit now points to the fixes later — engineered features
and kernels (SVM, chapter 05) and the recursive splits of decision trees (chapter 04).

A look ahead: NB 3 stops choosing weights by hand and writes down a single number for *how wrong* a set
of weights is — the **log-loss** — the surface that NB 4 then walks downhill.

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* Journal of the Royal Statistical
  Society B, 20(2), 215–242. DOI: 10.1111/j.2517-6161.1958.tb00292.x
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
  (§4.3). DOI: 10.1007/978-1-0716-1418-1
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (§4.4).
  DOI: 10.1007/978-0-387-84858-7

---

*Previous: 01 — From a linear score to a probability*  ·  *Next: 03 — Fitting I: what we optimize
(log-loss)*